In [3]:
from pathlib import Path
import json
import sys
import time
import uuid

import imageio.v2 as imageio
import numpy as np

REPO_ROOT = Path('/home/mzoellner/Projects/research/guided-diffusion')
sys.path.insert(0, str(REPO_ROOT / 'robomimic'))
sys.path.insert(0, str(REPO_ROOT / 'calvin' / 'calvin_env'))

import robomimic.envs
import robomimic.utils.env_utils as EnvUtils
import robomimic.utils.file_utils as FileUtils

CONFIG_PATH = REPO_ROOT / 'calvin_experiments' / 'env_playground_config.json'
SCENE_INDEX = {
    'sliding_door': 0,
    'drawer': 1,
    'button': 2,
    'switch': 3,
    'lightbulb': 4,
    'green_light': 5,
}


In [4]:
with open(CONFIG_PATH, 'r') as f:
    exp_cfg = json.load(f)

dataset_path = REPO_ROOT / exp_cfg['dataset_path']
output_root = REPO_ROOT / exp_cfg['output_root']
run_id = time.strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8]
out_dir = output_root / run_id
out_dir.mkdir(parents=True, exist_ok=True)
video_path = out_dir / 'robomimic_static_actions.mp4'

# Robomimic-centric env construction: load CALVIN env metadata from the HDF5,
# then let robomimic create EnvGym/PlayTableSimEnv exactly like training/eval.
env_meta = FileUtils.get_env_metadata_from_dataset(str(dataset_path))
env = EnvUtils.create_env_from_metadata(
    env_meta=env_meta,
    render=False,
    render_offscreen=True,
    use_image_obs=True,
)

try:
    obs = env.reset()
    state = env.get_state()
    scene = np.asarray(state['scene'], dtype=np.float32).copy()
    robot = np.asarray(state['robot'], dtype=np.float32).copy()

    for name, value in exp_cfg['env_setup'].items():
        scene[SCENE_INDEX[name]] = float(value)

    obs = env.reset_to({'scene': scene, 'robot': robot})

    frames = [env.render(mode='rgb_array', height=exp_cfg['render_height'], width=exp_cfg['render_width'])]
    actions = []

    for segment in exp_cfg['action_segments']:
        action = np.asarray(segment['action'], dtype=np.float32)
        for _ in range(int(segment['steps'])):
            obs, reward, done, info = env.step(action.copy())
            actions.append(action.copy())
            frames.append(env.render(mode='rgb_array', height=exp_cfg['render_height'], width=exp_cfg['render_width']))

    imageio.mimsave(video_path, frames, fps=int(exp_cfg['fps']))
    np.save(out_dir / 'actions.npy', np.stack(actions, axis=0))
    with open(out_dir / 'resolved_config.json', 'w') as f:
        json.dump(exp_cfg, f, indent=2)

    print(f'Wrote video: {video_path}')
    print(f'Applied env_setup: {exp_cfg["env_setup"]}')
finally:
    env.env.close()


argv[0]=--width=200
argv[1]=--height=200
Loaded EGL 1.5 after reload.
GL_VENDOR=AMD
GL_RENDERER=AMD Radeon Graphics (radeonsi, rembrandt, LLVM 15.0.7, DRM 3.64, 6.17.9-76061709-generic)
GL_VERSION=4.6 (Core Profile) Mesa 25.1.5-1pop0~1756399231~22.04~b84bab8
GL_SHADING_LANGUAGE_VERSION=4.60
Version = 4.6 (Core Profile) Mesa 25.1.5-1pop0~1756399231~22.04~b84bab8
Vendor = AMD
Renderer = AMD Radeon Graphics (radeonsi, rembrandt, LLVM 15.0.7, DRM 3.64, 6.17.9-76061709-generic)


EGL device choice: -1 of 2.


Created environment with name PlayTableSimEnv
Action size is 7
ROBOMIMIC WARNING(
    No environment version found in dataset!
    Cannot verify if dataset and installed environment versions match
)
disconnecting id -1 from server
ven = AMD
ven = AMD


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (200, 200) to (208, 208) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Wrote video: /home/mzoellner/Projects/research/guided-diffusion/outputs/calvin/base_tests/20260501_021859_23a4e939/robomimic_static_actions.mp4
Applied env_setup: {'drawer': 0.0, 'button': 0.0, 'switch': 0.0, 'lightbulb': 0.0, 'green_light': 0.0}
disconnecting id 0 from server
Destroy EGL OpenGL window.


[swscaler @ 0x1acef600] Warning: data is not aligned! This can lead to a speed loss


In [5]:
from IPython.display import Video, display

print(f'Video path: {video_path}')
display(Video(filename=str(video_path), embed=True))

print('Available image perspectives in this CALVIN/robomimic setup:')
print('- third_person: static external camera, saved by this notebook via env.render(...)')
print('- eye_in_hand: wrist / gripper camera, available in obs but not saved in this video')


Video path: /home/mzoellner/Projects/research/guided-diffusion/outputs/calvin/base_tests/20260501_021859_23a4e939/robomimic_static_actions.mp4


Available image perspectives in this CALVIN/robomimic setup:
- third_person: static external camera, saved by this notebook via env.render(...)
- eye_in_hand: wrist / gripper camera, available in obs but not saved in this video
